# Audit your ranker

**Run this after the session.** It needs `audit/labels.csv`, which is added to the repo after the workshop. Run `git pull` first.

The labels give each applicant a **group** (a synthetic protected attribute, A or B) and a **true fit** score (1 to 5) from an expert review of what each applicant actually did.

In [ ]:
from pathlib import Path
import pandas as pd

# Works whether you start Jupyter from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA, OUTPUTS = ROOT / "data", ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

In [ ]:
labels_path = ROOT / "audit" / "labels.csv"
results_path = OUTPUTS / "results.csv"
assert results_path.exists(), "Run 01_build_challenge.ipynb first to create outputs/results.csv"
assert labels_path.exists(), "audit/labels.csv not found yet. Run `git pull` to get it."

df = pd.read_csv(results_path).merge(pd.read_csv(labels_path), on="candidate_id")
df["shortlisted"] = df["shortlisted"].fillna(False).astype(bool)
df.groupby("group").size().rename("applicants")

## The four-fifths rule

For each group: **selection rate = selected / applicants**.
**Impact ratio = lower rate / higher rate.** A ratio under **0.80** flags possible adverse impact. It is a screening heuristic, not proof of discrimination.

In [ ]:
def four_fifths(frame: pd.DataFrame, selected_col: str) -> pd.DataFrame:
    rates = frame.groupby("group")[selected_col].agg(applicants="size", selected="sum")
    rates["selection_rate"] = rates["selected"] / rates["applicants"]
    rates["impact_ratio"] = rates["selection_rate"] / rates["selection_rate"].max()
    return rates.round(3)


for stage, col in [("R2 filter (passed)", "passed_filter"), ("R3 short list (top 10)", "shortlisted")]:
    table = four_fifths(df, col)
    flag = "possible adverse impact" if table["impact_ratio"].min() < 0.8 else "no flag"
    print(f"\n{stage}: lowest impact ratio {table['impact_ratio'].min():.2f} -> {flag}")
    display(table)

## Did the short list find the best people?

Compare your top 10 with the expert's true fit scores.

In [ ]:
top = df[df["shortlisted"]]
strong = df[(df["true_fit"] >= 4) & df["passed_filter"]]
missed = strong[~strong["shortlisted"]]

print(f"Your top 10 average true fit: {top['true_fit'].mean():.2f}")
print(f"Strong applicants (true fit 4-5) who passed R2: {len(strong)}")
print(f"Strong applicants you missed, by group:")
missed.groupby("group").size()

## Discuss

- If your impact ratio is under 0.80, what in your pipeline caused it? Look at the resumes of strong applicants you missed.
- Would you ship this? What would you change first?
- What would you measure every week once it is live?